In [2]:
import pyodbc
from ollama import chat

# ============================================================
# 1. SQL SERVER CONNECTION
# ============================================================

connection = pyodbc.connect(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=DESKTOP-NSQVUD1;"
    "DATABASE=orderssystem;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

cursor = connection.cursor()


# ============================================================
# 2. REVENUE ANALYSIS AGENT
# ============================================================

def revenue_analysis_agent(user_prompt):

    # --------------------------------------------------------
    # 1. OVERALL REVENUE SUMMARY
    # --------------------------------------------------------

    cursor.execute("""
        SELECT
            COUNT(orderid) AS total_orders,
            SUM(amount) AS total_revenue,
            AVG(amount) AS average_order_value,
            MIN(amount) AS minimum_order,
            MAX(amount) AS maximum_order
        FROM orders
    """)

    overall_row = cursor.fetchone()

    overall_data = f"""
Total Orders: {overall_row.total_orders}
Total Revenue: {overall_row.total_revenue}
Average Order Value: {overall_row.average_order_value}
Minimum Order: {overall_row.minimum_order}
Maximum Order: {overall_row.maximum_order}
"""


    # --------------------------------------------------------
    # 2. YEAR-WISE REVENUE
    # --------------------------------------------------------

    cursor.execute("""
        SELECT
            YEAR(dateofpurchase) AS revenue_year,
            COUNT(orderid) AS total_orders,
            SUM(amount) AS total_revenue,
            AVG(amount) AS average_order_value
        FROM orders
        GROUP BY YEAR(dateofpurchase)
        ORDER BY revenue_year
    """)

    year_rows = cursor.fetchall()

    year_data = "\n".join(
        f"Year: {r.revenue_year}, "
        f"Orders: {r.total_orders}, "
        f"Revenue: {r.total_revenue}, "
        f"Average Order: {r.average_order_value}"
        for r in year_rows
    )


    # --------------------------------------------------------
    # 3. MONTH-WISE REVENUE
    # --------------------------------------------------------

    cursor.execute("""
        SELECT
            YEAR(dateofpurchase) AS revenue_year,
            MONTH(dateofpurchase) AS revenue_month,
            COUNT(orderid) AS total_orders,
            SUM(amount) AS total_revenue,
            AVG(amount) AS average_order_value
        FROM orders
        GROUP BY
            YEAR(dateofpurchase),
            MONTH(dateofpurchase)
        ORDER BY
            revenue_year,
            revenue_month
    """)

    month_rows = cursor.fetchall()

    month_data = "\n".join(
        f"Year: {r.revenue_year}, "
        f"Month: {r.revenue_month}, "
        f"Orders: {r.total_orders}, "
        f"Revenue: {r.total_revenue}, "
        f"Average Order: {r.average_order_value}"
        for r in month_rows
    )


    # --------------------------------------------------------
    # 4. CUSTOMER-WISE REVENUE
    # --------------------------------------------------------

    cursor.execute("""
        SELECT
            c.custid,
            c.custname,
            COUNT(o.orderid) AS total_orders,
            SUM(o.amount) AS total_revenue,
            AVG(o.amount) AS average_order_value
        FROM orders o
        INNER JOIN customer c
            ON c.custid = o.custid
        GROUP BY
            c.custid,
            c.custname
        ORDER BY total_revenue DESC
    """)

    customer_rows = cursor.fetchall()

    customer_data = "\n".join(
        f"Customer ID: {r.custid}, "
        f"Customer: {r.custname}, "
        f"Orders: {r.total_orders}, "
        f"Revenue: {r.total_revenue}, "
        f"Average Order: {r.average_order_value}"
        for r in customer_rows
    )


    # --------------------------------------------------------
    # 5. SALESMAN-WISE REVENUE
    # --------------------------------------------------------

    cursor.execute("""
        SELECT
            s.salesmanid,
            s.salesmanname,
            COUNT(o.orderid) AS total_orders,
            SUM(o.amount) AS total_revenue,
            AVG(o.amount) AS average_order_value
        FROM orders o
        INNER JOIN salesman s
            ON s.salesmanid = o.salesmanid
        GROUP BY
            s.salesmanid,
            s.salesmanname
        ORDER BY total_revenue DESC
    """)

    salesman_rows = cursor.fetchall()

    salesman_data = "\n".join(
        f"Salesman ID: {r.salesmanid}, "
        f"Salesman: {r.salesmanname}, "
        f"Orders: {r.total_orders}, "
        f"Revenue: {r.total_revenue}, "
        f"Average Order: {r.average_order_value}"
        for r in salesman_rows
    )


    # --------------------------------------------------------
    # 6. PRODUCT-WISE REVENUE
    # --------------------------------------------------------

    cursor.execute("""
        SELECT
            p.productid,
            p.productname,
            COUNT(o.orderid) AS total_orders,
            SUM(o.amount) AS total_revenue
        FROM orders o
        INNER JOIN product p
            ON p.productid = o.productid
        GROUP BY
            p.productid,
            p.productname
        ORDER BY total_revenue DESC
    """)

    product_rows = cursor.fetchall()

    product_data = "\n".join(
        f"Product ID: {r.productid}, "
        f"Product: {r.productname}, "
        f"Orders: {r.total_orders}, "
        f"Revenue: {r.total_revenue}"
        for r in product_rows
    )


    # --------------------------------------------------------
    # 7. REVENUE TREND
    # --------------------------------------------------------

    cursor.execute("""
        SELECT
            YEAR(dateofpurchase) AS revenue_year,
            MONTH(dateofpurchase) AS revenue_month,
            SUM(amount) AS total_revenue
        FROM orders
        GROUP BY
            YEAR(dateofpurchase),
            MONTH(dateofpurchase)
        ORDER BY
            revenue_year,
            revenue_month
    """)

    trend_rows = cursor.fetchall()

    trend_data = "\n".join(
        f"{r.revenue_year}-{r.revenue_month:02d}: "
        f"Revenue = {r.total_revenue}"
        for r in trend_rows
    )


    # ========================================================
    # 8. SEND DATA + USER PROMPT TO OLLAMA
    # ========================================================

    response = chat(
        model="llama3.2",

        messages=[

            {
                "role": "system",
                "content": """
You are an AI Revenue Analysis Agent.

Analyze the business revenue using ONLY the
SQL Server data supplied by the program.

You can answer questions about:

- Total revenue
- Year-wise revenue
- Month-wise revenue
- Revenue growth
- Revenue decline
- Customer revenue
- Salesman revenue
- Product revenue
- Average order value
- Highest revenue
- Lowest revenue
- Revenue trends
- Year-on-year comparison
- Month-on-month comparison
- Revenue contribution
- Management revenue reporting

IMPORTANT RULES:

1. Use only the supplied database information.

2. Never invent database records.

3. Never invent revenue figures.

4. Never invent customers.

5. Never invent salesmen.

6. Never invent products.

7. If the user asks for a calculation,
   calculate it using the supplied data.

8. Revenue growth percentage:

   ((Current Revenue - Previous Revenue)
   / Previous Revenue) * 100

9. If previous revenue is zero,
   explain that percentage growth cannot
   be calculated.

10. Revenue is NOT profit.

11. Do not call revenue profit.

12. If information is unavailable,
    clearly say that it is unavailable.

13. Answer the user's question directly.

14. Do not answer questions that cannot
    be supported by the supplied database.

15. Use actual years and months from
    the database.
"""
            },

            {
                "role": "user",
                "content": f"""
============================================================
OVERALL REVENUE
============================================================

{overall_data}


============================================================
YEAR-WISE REVENUE
============================================================

{year_data}


============================================================
MONTH-WISE REVENUE
============================================================

{month_data}


============================================================
CUSTOMER-WISE REVENUE
============================================================

{customer_data}


============================================================
SALESMAN-WISE REVENUE
============================================================

{salesman_data}


============================================================
PRODUCT-WISE REVENUE
============================================================

{product_data}


============================================================
REVENUE TREND
============================================================

{trend_data}


============================================================
USER QUESTION
============================================================

{user_prompt}


Answer the user's question using only
the supplied SQL Server data.
"""
            }

        ]
    )

    return response.message.content


# ============================================================
# 3. USER INTERFACE
# ============================================================

print("\n==============================================")
print("          AI REVENUE ANALYSIS AGENT")
print("==============================================")

print("\nYou can ask ANY revenue-related question.")
print("Examples:")
print("  Which customer generated the highest revenue?")
print("  What is the total revenue?")
print("  Compare revenue between 2025 and 2026.")
print("  Which product generated the most revenue?")
print("  Which salesman generated the most revenue?")
print("  What was the average order value?")
print("  Which month had the highest revenue?")
print("  Calculate year-on-year revenue growth.")
print("\nType 'exit' to quit.")


# ============================================================
# 4. CONTINUOUS USER INPUT
# ============================================================

while True:

    user_prompt = input(
        "\nEnter your revenue question: "
    )

    # Remove leading/trailing spaces
    user_prompt = user_prompt.strip()

    # Exit
    if user_prompt.lower() == "exit":
        print("\nExiting Revenue Agent...")
        break

    # Empty input
    if user_prompt == "":
        print("Please enter a revenue question.")
        continue

    try:

        print("\nAnalyzing database...")

        result = revenue_analysis_agent(user_prompt)

        print("\n==============================================")
        print("             AI REVENUE ANALYSIS")
        print("==============================================")

        print(result)

        print("==============================================")

    except Exception as e:

        print("\nERROR:")
        print("----------------------------------------------")
        print(e)


# ============================================================
# 5. CLOSE CONNECTION
# ============================================================

cursor.close()
connection.close()

print("\nDatabase connection closed.")
print("Revenue Agent stopped.")


          AI REVENUE ANALYSIS AGENT

You can ask ANY revenue-related question.
Examples:
  Which customer generated the highest revenue?
  What is the total revenue?
  Compare revenue between 2025 and 2026.
  Which product generated the most revenue?
  Which salesman generated the most revenue?
  What was the average order value?
  Which month had the highest revenue?
  Calculate year-on-year revenue growth.

Type 'exit' to quit.



Enter your revenue question:  Compare revenue between 2025 and 2026.



Analyzing database...

             AI REVENUE ANALYSIS
To compare revenue between 2025 and 2026, we need to calculate the total revenue for each year.

Total revenue in 2025:
SELECT SUM(R Revenue) FROM MonthlyRevenue WHERE Year = 2025

Total revenue in 2026:
SELECT SUM(R.Revenue) FROM MonthlyRevenue WHERE Year = 2026

Let's calculate the total revenue for each year:

Total revenue in 2025:
SELECT SUM(R.Revenue) FROM MonthlyRevenue WHERE Year = 2025
= 32000.00 + 18500.00 + 24500.00 + 16000.00 + 29500.00
= 115500.00

Total revenue in 2026:
SELECT SUM(R.Revenue) FROM MonthlyRevenue WHERE Year = 2026
= 22000.00 + 17500.00 + 26500.00 + 19500.00 + 31000.00 + 14500.00 + 28500.00 + 17000.00 + 23500.00
= 184000.00

Now, let's compare the revenue between 2025 and 2026:
184000.00 (2026) - 115500.00 (2025) = 68500.00 (Increase in revenue)

The revenue increased by 68500.00 between 2025 and 2026.



Enter your revenue question:  What was the average order value?



Analyzing database...

             AI REVENUE ANALYSIS
SELECT Average Order Value FROM RevenueData


KeyboardInterrupt: Interrupted by user